# Part 1

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import math

In [6]:
m = 10
mean_time = 1
mean_service = 8
n_reps = 10
n_customers = 10000
np.random.seed(42)

def erlang_b(A,m):
    num = (A**m) / math.factorial(m)
    den = sum((A**i) /  math.factorial(i) for i in range(m + 1))
    return num/den

def compute_ci(data, alpha = 0.05):
    n = len(data)
    mean = np.mean(data)
    std_err = np.std(data, ddof = 1) / np.sqrt(n)
    t_crit = stats.t.ppf(1 - alpha/2, df = n-1)
    lower = mean - t_crit * std_err
    upper = mean + t_crit *std_err
    return mean, lower, upper

def simulate_loss_system(generate_arrivals, generate_services):
    blocking_fraction = []

    for _  in range(n_reps):
        inter_arrivals = generate_arrivals(n_customers)
        service_times = generate_services(n_customers)
        arrival_times = np.cumsum(inter_arrivals)

        servers_free_time = np.zeros(m)
        blocked = 0

        for i in range(n_customers):
            arr = arrival_times[i]

            free_servers = servers_free_time <= arr

            if np.any(free_servers):
                idx = np.argmax(free_servers)
                servers_free_time[idx] = arr + service_times[i]
            else:
                blocked += 1

        blocking_fraction.append(blocked / n_customers)
    
    mean, lower, upper = compute_ci(blocking_fraction)
    return mean, lower, upper


def arr_poisson(n):
    return np.random.exponential(scale=mean_time, size = n)

def srv_exp(n):
    return np.random.exponential(scale = mean_service, size = n)

mean_mm, low_mm, up_mm = simulate_loss_system(arr_poisson, srv_exp)

print(f"Simulated Blocking: {mean_mm:.4f}  (95% CI: [{low_mm:.4f}, {up_mm:.4f}])")
print(f"Exact Erlang-B:     {erlang_b(mean_time * mean_service, m):.4f}")


Simulated Blocking: 0.1233  (95% CI: [0.1176, 0.1290])
Exact Erlang-B:     0.1217


# Part2

In [8]:
np.random.seed(42)
def erlang_array(n):
    return np.random.gamma(shape = 2, scale = 0.5, size = n)

def hyper_inter_exp(n):
    coin = np.random.rand(n)

    return np.where(coin <=0.8, 
                    np.random.exponential(1/0.8333, size = n),
                    np.random.exponential(1/5, size = n))

mean_erlang, low_erlang, up_erlang = simulate_loss_system(erlang_array, srv_exp)
mean_hyper, low_hyper, up_hyper = simulate_loss_system(hyper_inter_exp, srv_exp)

print(f"Erlang Arrivals Blocking:     {mean_erlang:.4f}  (95% CI: [{low_erlang:.4f}, {up_erlang:.4f}])")
print(f"Hyper-Exp Arrivals Blocking:  {mean_hyper:.4f}  (95% CI: [{low_hyper:.4f}, {up_hyper:.4f}])")

Erlang Arrivals Blocking:     0.0922  (95% CI: [0.0880, 0.0964])
Hyper-Exp Arrivals Blocking:  0.1417  (95% CI: [0.1364, 0.1471])


# Part 3

In [13]:
np.random.seed(42)

def service_constant(n):
    return np.full(n, mean_service)

def service_pareto1(n):
    k = 1.05
    beta = mean_service * (k - 1) / k
    pareto = (np.random.pareto(k, size = n) + 1) * beta
    return pareto

def service_pareto2(n):
    k = 2.05
    beta = mean_service * (k - 1) / k
    pareto = (np.random.pareto(k, size = n) + 1) * beta
    return pareto


def srv_erlang4(n):
    erlang = np.random.gamma(shape=4, scale=2.0, size=n)
    return erlang

def srv_uniform(n):
    service = np.random.uniform(low=0.0, high=2.0 * mean_service, size=n)
    return service


mean_con, low_con, up_con = simulate_loss_system(arr_poisson, service_constant)

mean_p1, low_p1, up_p1 = simulate_loss_system(arr_poisson, service_pareto1)
mean_p2, low_p2, up_p2 = simulate_loss_system(arr_poisson, service_pareto2)

# Run the simulations using the core logic defined in Cell 2
mean_erl4, low_erl4, up_erl4 = simulate_loss_system(arr_poisson, srv_erlang4)
mean_uni, low_uni, up_uni = simulate_loss_system(arr_poisson, srv_uniform)

print(f"Constant Service Blocking:    {mean_con:.4f}  (95% CI: [{low_con:.4f}, {up_con:.4f}])")

print(f"Pareto (k=2.05) Blocking:     {mean_p2:.4f}  (95% CI: [{low_p2:.4f}, {up_p2:.4f}])")
print(f"Pareto (k=1.05) Blocking:     {mean_p1:.4f}  (95% CI: [{low_p1:.4f}, {up_p1:.4f}])")

print(f"Erlang-4 Service Blocking:  {mean_erl4:.4f}  (95% CI: [{low_erl4:.4f}, {up_erl4:.4f}])")
print(f"Uniform Service Blocking:   {mean_uni:.4f}  (95% CI: [{low_uni:.4f}, {up_uni:.4f}])")

print(f"Exact Erlang-B:     {erlang_b(mean_time * mean_service, m):.4f}")



Constant Service Blocking:    0.1225  (95% CI: [0.1183, 0.1268])
Pareto (k=2.05) Blocking:     0.1207  (95% CI: [0.1151, 0.1264])
Pareto (k=1.05) Blocking:     0.0008  (95% CI: [0.0002, 0.0015])
Erlang-4 Service Blocking:  0.1225  (95% CI: [0.1184, 0.1266])
Uniform Service Blocking:   0.1211  (95% CI: [0.1185, 0.1237])
Exact Erlang-B:     0.1217
